# Stage 1 — LIME Feature Extraction

**Reads:** Input texts defined in this notebook  
**Writes:** `outputs/lime_results.json`

Each record saved:
```json
{
  "text": "...",
  "fused_text": "...",
  "lime_features": [["hypertension", 0.42], ...]
}
```

> **Tip:** Run cells top to bottom on first run.  
> If `outputs/lime_results.json` already exists the checkpoint cell will warn you — set `FORCE_RERUN = True` to overwrite it.

## 1. Imports

In [1]:
from lime.lime_text import LimeTextExplainer

from config import (
    CLASS_NAMES,
    LIME_NUM_FEATURES,
    LIME_NUM_SAMPLES,
    LIME_RESULTS_PATH,
)
from model_loaders import load_classifier, load_ner_pipeline
from pipeline_helpers import (
    checkpoint_exists,
    load_checkpoint,
    make_lime_predictor,
    merge_entities,
    save_checkpoint,
)

## 2. Configuration

In [2]:
# Set True to re-run even if lime_results.json already exists
FORCE_RERUN = True

# ── Add / edit your input texts here ────────────────────────────────────────
# with open("test_data.txt", "r", encoding="utf-8") as f:
#     full_text = [line.strip() for line in f if line.strip()]

# # First 200 texts
# TEXTS = full_text[:50]

TEXTS = ["Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive ascites associated with endometriosis has been reported in rare cases, this patient was also noted to have massive destruction of the pelvic peritoneum. Failure of medical suppression necessitated total abdominal hysterectomy and bilateral salpingo-oophorectomy. Several months after surgery ascites resolved, possibly with reestablishment of the pelvic peritoneum. ",]


In [3]:
print(TEXTS)

['Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive ascites associated with endometriosis has been reported in rare cases, this patient was also noted to have massive destruction of the pelvic peritoneum. Failure of medical suppression necessitated total abdominal hysterectomy and bilateral salpingo-oophorectomy. Several months after surgery ascites resolved, possibly with reestablishment of the pelvic peritoneum. ']


## 3. Checkpoint check

In [4]:
if checkpoint_exists(LIME_RESULTS_PATH) and not FORCE_RERUN:
    print(f"⚠️  Checkpoint found at '{LIME_RESULTS_PATH}'.")
    print("    Set FORCE_RERUN = True in the cell above to overwrite.")
    print("    Loading existing results …")
    results = load_checkpoint(LIME_RESULTS_PATH)
else:
    results = None
    print("No checkpoint found (or FORCE_RERUN=True). Will run LIME.")

No checkpoint found (or FORCE_RERUN=True). Will run LIME.


## 4. Load models

In [5]:
if results is None:
    classifier_model, classifier_pipeline = load_classifier()
    ner_pipeline = load_ner_pipeline()

[Loader] Loading classifier from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Models/my_medical_model' …


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[Loader] Classifier ready.

[Loader] Loading NER model 'd4data/biomedical-ner-all' …


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[Loader] NER model ready.



## 5. Run LIME

In [6]:
if results is None:
    explainer = LimeTextExplainer(class_names=CLASS_NAMES)
    predictor = make_lime_predictor(classifier_model, classifier_pipeline)

    results = []
    for i, text in enumerate(TEXTS):
        print(f"\nProcessing text {i + 1}/{len(TEXTS)} …")

        # Fuse multi-word biomedical entities before LIME
        fused_text = merge_entities(text, ner_pipeline)
        print(f"  Fused text preview: {fused_text[:120]}…")

        exp = explainer.explain_instance(
            fused_text,
            predictor,
            num_features=LIME_NUM_FEATURES,
            num_samples=LIME_NUM_SAMPLES,
        )

        # Restore underscores → spaces for downstream readability
        lime_features = [
            [w.replace("_", " "), float(score)]
            for w, score in exp.as_list()
        ]

        print(f"  Top features: {[f[0] for f in lime_features]}")

        results.append({
            "text":          text,
            "fused_text":    fused_text,
            "lime_features": lime_features,
        })

    print("\n✅ LIME complete.")


Processing text 1/1 …
  Fused text preview: Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive ascites associated with…
  Top features: ['ascites', 'peritoneum', 'associated', 'abdominal', 'resolved', 'Although']

✅ LIME complete.


## 6. Inspect results

In [7]:
for i, r in enumerate(results):
    print(f"\n── Text {i + 1} ──────────────────────────")
    print(f"Text preview : {r['text'][:100]}…")
    print(f"Top features : {r['lime_features']}")


── Text 1 ──────────────────────────
Text preview : Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive asc…
Top features : [['ascites', 0.2069267426665309], ['peritoneum', 0.1668607133109441], ['associated', 0.040685885129984035], ['abdominal', 0.03312089740404215], ['resolved', 0.028189553883372465], ['Although', 0.023256197121805124]]


## 7. Save checkpoint

In [8]:
save_checkpoint(results, LIME_RESULTS_PATH)
print(f"\n➡️  Continue to notebook 02_ontology.ipynb")

[Checkpoint] Saved 1 records → 'Pivot_OP\lr.json'

➡️  Continue to notebook 02_ontology.ipynb
